# Визуализация данных. Анализ данных

© Петров М.В., старший преподаватель кафедры киберфотоники, Самарский университет

# Лекция 6. Задачи классификации. Метрики качества. Классификация kNN

## Содержание

1. [Введение](#61-введение)
2. [Метод $k$-ближайших соседей](#62-метод--ближайших-соседей)
3. [Датасет `Rain in Australia` для бинарной классификации](#63-датасет-rain-in-australia-для-бинарной-классификации)
4. [Библиотека `missingno`: визуальный анализ пропусков в данных](#64-библиотека-missingno-визуальный-анализ-пропусков-в-данных)
5. [Подготовка данных](#65-подготовка-данных)
6. [Кодирование признака](#66-кодирование-признака)
7. [Подготовка данных (продолжение)](#67-подготовка-данных-продолжение)
8. [Обучение и предсказание](#68-обучение-и-предсказание)
9. [Оценка качества модели](#69-оценка-качества-модели)
10. [Метрики оценки точности бинарной классификации](#610-метрики-оценки-точности-бинарной-классификации)

## 6.1 Введение

Гайды:
- [Открытый курс машинного обучения. Тема 3. Классификация, деревья решений и метод ближайших соседей](https://habr.com/ru/companies/ods/articles/322534/)

Классическое, общее определение машинного обучения звучит так (T. Mitchell "Machine learning", 1997):
> Говорят, что компьютерная программа обучается при решении какой-то задачи из класса $T$, если ее производительность, согласно метрике $P$, улучшается при накоплении опыта $E$.

Среди самых популярных задач $T$ в машинном обучении:
- Классификация &ndash; отнесение объекта к одной из категорий на основании его признаков.
- Регрессия &ndash; прогнозирование количественного признака объекта на основании прочих его признаков.
- Кластеризация &ndash; разбиение множества объектов на группы на основании признаков этих объектов так, чтобы внутри групп объекты были похожи между собой, а вне одной группы &ndash; менее похожи.
- Нахождение аномалий &ndash; поиск объектов, "сильно непохожих" на все остальные в выборке, либо на какую-то группу объектов.
- Другие задачи, более специфичные. Хороший обзор дан в главе "Machine Learning basics" книги ["Deep Learning": Ian Goodfellow, Yoshua Bengio, Aaron Courville, 2016](http://www.deeplearningbook.org/).

Под опытом $E$ понимаются данные, и в зависимости от данных алгоритмы машинного обучения разделяются на 2 вида:
- обучение с учителем &ndash; `supervised learning`;
- обучение без учителя &ndash; `unsupervised learning`.

В задачах обучения без учителя имеется выборка, состоящая из объектов, описываемых набором признаков. В задачах обучения с учителем вдобавок к этому для каждого объекта некоторой выборки, называемой *обучающей*, известен *целевой признак* &ndash; по сути это то, что хотелось бы прогнозировать для прочих объектов, не из обучающей выборки.

#### Пример

Задачи классификации и регрессии &ndash; это задачи обучения с учителем. В качестве примера будем представлять задачу кредитного скоринга: на основе накопленных кредитной организацией данных о своих клиентах хочется прогнозировать *невозврат кредита*. Здесь для алгоритма опыт $E$ &ndash; это имеющаяся обучающая выборка: набор объектов (людей), каждый из которых характеризуется набором признаков (таких как *возраст*, *зарплата*, *тип кредита*, *невозвраты в прошлом* и т.д.), а также целевым признаком. Если этот целевой признак &ndash; просто факт *невозврата кредита* ($1$ или $0$, т.е. банк знает о своих клиентах, кто вернул кредит, а кто &ndash; нет), то это задача (бинарной) ***классификации***. Если известно, *на сколько по времени клиент затянул с возвратом кредита*, и хочется то же самое прогнозировать для новых клиентов, то это будет задачей ***регрессии***.

## 6.2 Метод $k$-ближайших соседей

Метод $k$-ближайших соседей (*`k-nearest neighbors algorithm`*, *`k-NN`*) &ndash; метрический алгоритм для автоматической классификации объектов или регрессии.

Алгоритм выполняет поиск заранее заданного количества кластеров в немаркированном многомерном наборе данных. Достигается это с помощью простого представления о том, что такое оптимальная кластеризация.
- «Центр кластера» &ndash; арифметическое среднее всех точек, относящихся к этому кластеру.
- Каждая точка ближе к центру своего кластера, чем к центрам других кластеров.

В случае использования метода для классификации объект присваивается тому классу, который является наиболее распространённым среди $k$ соседей данного элемента, классы которых уже известны. Алгоритм может быть применим к выборкам с большим количеством атрибутов (многомерным). Для этого перед применением нужно определить функцию расстояния; классический вариант такой функции &ndash; *евклидова метрика*.

Разные атрибуты могут иметь разный диапазон представленных значений в выборке (например атрибут $А$ представлен в диапазоне от $0.1$ до $0.5$, а атрибут $Б$ представлен в диапазоне от $1000$ до $5000$), то значения дистанции могут сильно зависеть от атрибутов с б*о́*льшими диапазонами. Поэтому данные обычно подлежат *нормализации*.

### Датасет &ndash; Ирисы Фишера

[Ирисы Фишера](https://ru.wikipedia.org/wiki/%D0%98%D1%80%D0%B8%D1%81%D1%8B_%D0%A4%D0%B8%D1%88%D0%B5%D1%80%D0%B0) состоят из данных о $150$ экземплярах ириса, по $50$ экземпляров из трёх видов &ndash; `Ирис щетинистый` (`Iris setosa`), `Ирис виргинский` (`Iris virginica`) и `Ирис разноцветный` (`Iris versicolor`).
Для каждого экземпляра измерялись четыре характеристики (в сантиметрах):
- Длина наружной доли околоцветника (англ. `sepal length`);
- Ширина наружной доли околоцветника (англ. `sepal width`);
- Длина внутренней доли околоцветника (англ. `petal length`);
- Ширина внутренней доли околоцветника (англ. `petal width`).

На основании этого набора данных требуется построить правило классификации, определяющее *вид растения* по данным измерений. Это задача *многоклассовой классификации*, так как имеется три класса &ndash; три вида ириса.
Один из классов (`Iris setosa`) линейно-разделим от двух остальных.

> Необходимо установить [scikit-learn](https://scikit-learn.org/stable/):
> 
> ```bash
> uv pip install -U scikit-learn
> ```

In [ ]:
import sklearn
from sklearn import datasets

dataset = datasets.load_iris()
print(sklearn.__version__)
print(list(dataset.keys()))

Значение ключа `DESCR` &ndash; это краткое описание набора данных:

In [ ]:
print(dataset['DESCR'])

Значение ключа `target_names` &ndash; массив меток классов (в данном случае сорта цветов):

In [ ]:
list(dataset['target_names'])

Список названий полей-признаков:

In [ ]:
list(dataset['feature_names'])

Сами признаки записаны в numpy-массиве `data`:

In [ ]:
type(dataset['data']), dataset['data'].shape

Первые пять строк массива `data`:

In [ ]:
dataset['data'][:5]

«Ответы» записаны в `target`:

In [ ]:
type(dataset['target']), dataset['target'].shape

Числовым значениям в `target` соответствуют названия в `target_names`: 0 &ndash; `setosa`, 1 &ndash; `versicolor`, а 2 &ndash; `virginica`:

In [ ]:
print(dataset['target'])

### Разделение выборки на обучающую и тестовую

Разобьём весь датасет на две части. Одна часть данных используется для построения модели машинного обучения и называется *обучающими данными* (*training data*) или *обучающим набором* (*training set*). Остальные данные будут использованы для оценки качества модели, их называют *тестовыми данными* (*test data*), *тестовым набором* (*test set*) или *контрольным набором* (*hold-out set*).

<div align="center">
  <img src="images/model_train_test.svg" width="500" title="Обучение и тестирование модели"/>
  <p style="text-align: center">
    Рисунок 1 &ndash; Обучение и тестирование модели
  </p>
</div>

> Также для некоторых моделей часть данных из обучающей выборки выделяется в *валидационный набор*, который используется для вычисления метрик во время обучения.

Обычно отбирают в обучающий набор $70\text{-}80\%$ строк данных, и оставшиеся $\%$ объявляются тестовым набором.  
В библиотеке `scikit-learn` есть функция `train_test_split`, которая перемешивает набор данных и разбивает его на две части.

In [ ]:
from sklearn.model_selection import train_test_split
ex_X, ex_y = dataset.data, dataset.target
# Для упрощения визуализации оставим только 2 признака
ex_X = ex_X[:, :2]

ex_X_train, ex_X_test, ex_y_train, ex_y_test = train_test_split(
    ex_X, ex_y, stratify=ex_y, train_size=0.75, random_state=42
)

In [ ]:
print(f"X_train shape: {ex_X_train.shape}")
print(f"y_train shape: {ex_y_train.shape}")
print(f"X_test shape: {ex_X_test.shape}")
print(f"y_test shape: {ex_y_test.shape}")

Аргументы в `train_test_split`:
```python
ex_X_train, ex_X_test, ex_y_train, ex_y_test = train_test_split(
    ex_X, ex_y, stratify=ex_y, train_size=0.75, random_state=42
)
```
- `stratify` &ndash; данный параметр отвечает за пропорциональное распределение сэмплов в обучающей и тестовой выборке относительно выбранного категориального признака.
  
  > Если в исходной выборке у бинарного признака `F` $75\%$ сэмплов &ndash; класс `A`, а $25\%$ сэмплов &ndash; класс `B`, то это же соотношение в метках будет и в обучающей и тестовой выборках после разделения.
- `train_size` &ndash; задает долю сэмплов в обучающей выборке (`test_size` задают долю сэмплов в тестовой).
- `random_state` &ndash; значение зерна (*seed*) для генератора случайных чисел.  
   
   > Использование одного и того же значения `random_state` позволяет получать одинаковое разбиение данных при повторном запуске на одной и той же машине (как минимум, как максимум &ndash; на устройствах с одной и той же версией ОС, либ и принадлежащих к одной аппаратной платформе [архитектуре]).

### Создание объекта алгоритма/модели-предсказателя

В `sklearn` все модели машинного обучения реализованы в собственных классах, называемых классами `Estimator`. Алгоритм классификации на основе метода k ближайших соседей реализован в классификаторе [KNeighborsClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html) модуля `sklearn.neighbors`. Прежде чем использовать эту модель, нам нужно создать объект-экземпляр класса, указав параметры модели. Самым важным параметром `KNeighborsClassifier` является количество соседей, которые мы установим равным $3$.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
ex_model_knn = KNeighborsClassifier(n_neighbors=3, metric='euclidean')

### Обучение модели

In [ ]:
ex_model_knn.fit(ex_X_train, ex_y_train)

### Визуализация примера работы алгоритма

In [ ]:
import pandas as pd
import numpy as np
# magic function - см. Interactive Plotting
%matplotlib widget
from ipywidgets import *
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from sklearn.metrics import DistanceMetric
from matplotlib.colors import Normalize, LinearSegmentedColormap, ListedColormap

# Тестовая точка - sepal length, width
test_point = [7, 3.5]
tpp = ex_model_knn.predict([test_point])

colors = ["#7C3DBF", "#956600", "#1db65a"]
scc = ListedColormap(colors)

# Ближайшие 6 соседей
n_d, n_i = ex_model_knn.kneighbors([test_point], n_neighbors=6, return_distance=True)
points = np.reshape(ex_X_train[n_i].ravel(), (-1, 2))

from sklearn.inspection import DecisionBoundaryDisplay
fig, ax = plt.subplots(1, 2, figsize=(8, 4), sharex=True, sharey=True)
ax[0].set_aspect('equal')
ax[1].set_aspect('equal')
sc = ax[0].scatter(ex_X_train[:, 0], ex_X_train[:, 1], c=ex_y_train, cmap=scc)
ax[0].legend(*sc.legend_elements())
ax[0].set_xlabel(dataset.feature_names[0])
ax[0].set_ylabel(dataset.feature_names[1])
ax[0].set_title(r"Добавление новой точки $(7, 3.5)$")

ax[0].plot(test_point[0], test_point[1], 'ro')
circle = plt.Circle((test_point), np.max(n_d), color='r', ls='--', fill=False)
ax[0].add_patch(circle)
for i in range(6):
    p = points[i, :]
    p = np.vstack((p, test_point))
    ax[0].plot(p[:, 0], p[:, 1], 'r--')

DecisionBoundaryDisplay.from_estimator(
        ex_model_knn,
        ex_X,
        alpha=0.25,
        ax=ax[1],
        response_method="predict",
        plot_method="pcolormesh",
        xlabel=dataset.feature_names[0],
        ylabel=dataset.feature_names[1],
        shading="auto",
        multiclass_colors=colors
    )
circle = plt.Circle((test_point), np.max(n_d), color=scc(tpp), ls='--', fill=False)
ax[1].add_patch(circle)
ax[1].scatter(ex_X_train[:, 0], ex_X_train[:, 1], c=ex_y_train, cmap=scc)
ax[1].legend(*sc.legend_elements())
ax[1].set_title(fr"Соответствие $(7, 3.5) \longleftrightarrow \text{{класс {tpp[0]}}}$")
for i in range(6):
    p = points[i, :]
    p = np.vstack((p, test_point))
    ax[1].plot(p[:, 0], p[:, 1], ls = '--', color = 'r')
ax[1].plot(test_point[0], test_point[1], marker='o', color=scc(tpp))
ax[0].set_xlim(ax[1].get_xlim())
ax[0].set_ylim(ax[1].get_ylim())
fig.tight_layout()
plt.show()

> `sharex`, `sharey` в `plt.subplots` «синхронизирует» пределы (min, max) осей на субграфиках, масштабирование и панорамирование так же применяется синхронно к субграфикам. Можно одну ось явно «привязать» к другой оси, при этом нельзя одну ось привязать к двум и более осям. Динамически отвязать ось в коде не получится.
> Подробнее:
> - [Shared axis](https://matplotlib.org/stable/gallery/subplots_axes_and_figures/shared_axis_demo.html)
> - [matplotlib.axes.Axes.sharex](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.sharex.html)
>
> Тот же самый функционал с явной привязкой в примере выше:
> ```python
> ax[1].sharex(ax[0])
> ```

На левом графике добавляем новую точку $(7, 3.5)$, не входящую ни в обучающую выборку, ни в тестовую.

- `tpp` &ndash; результат предсказания моделью точки `test_point = [7, 3.5]`
- `scc` &ndash; «color map», созданный из набора цветов `colors` (три цвета)
- `n_d` &ndash; расстояния до ближайших точек (6 соседей)
- `n_i` &ndash; индексы ближайших точек из обучающей выборки
- `points` &ndash; координаты ближайших соседей

`sklearn.inspection.DecisionBoundaryDisplay` &ndash; это класс в библиотеке `scikit-learn`, предназначенный для визуализации разделяющих поверхностей (*decision boundaries*) моделей классификации в двумерном пространстве признаков.  
Параметры `from_estimator()`:
- `estimator` &ndash; обученный классификатор
- `x` &ndash; данные `(n_samples, 2)` &ndash; поддерживаются строго двумерные данные (поскольку график строится на плоскости)
- `plot_method` &ndash; метод отрисовки:
  + 'contourf' (по умолчанию) &ndash; заполненные контуры
  + 'contour' &ndash; линии контуров
  + 'pcolormesh' &ndash; пиксельная псевдоцветная сетка
- `response_method` &ndash; что именно запрашивать у модели для покраски сетки:
  + 'predict' &ndash; предсказанные классы (четкие границы)
  + 'predict_proba' &ndash; вероятности классов (плавные переходы уверенности)
  + 'decision_function' &ndash; расстояние до разделяющей гиперплоскости
  + 'auto' &ndash; автоматический выбор в порядке приоритета выше

In [ ]:
tpp

In [ ]:
scc

In [ ]:
n_d

In [ ]:
n_i

In [ ]:
points

## 6.3 Датасет [Rain in Australia](https://www.kaggle.com/datasets/jsphyg/weather-dataset-rattle-package) для бинарной классификации

### Описание датасета

Датасет содержит данные о метеонаблюдениях в Австралии, цель &ndash; прогнозирование дождя на следующий день. Целевой признак &ndash; `RainTomorrow`.

| Признак       | Описание                                                                                               | Единицы измерения   |
|---------------|--------------------------------------------------------------------------------------------------------|---------------------|
| Location      | The common name of the location of the weather station                                                 |
| MinTemp       | Minimum temperature in the 24 hours to 9am. Sometimes only known to the nearest whole degree.          | degrees Celsius
| MaxTemp       | Maximum temperature in the 24 hours from 9am. Sometimes only known to the nearest whole degree.        | degrees Celsius
| Rainfall      | Precipitation (rainfall) in the 24 hours to 9am. Sometimes only known to the nearest whole millimetre. | millimetres
| Sunshine      | Bright sunshine in the 24 hours to midnight                                                            | hours
| WindGustDir   | Direction of strongest gust in the 24 hours to midnight                                                | 16 compass points
| WindGustSpeed | Speed of strongest wind gust in the 24 hours to midnight                                               | kilometres per hour
| WindDir9am    | Wind direction averaged over 10 minutes prior to 9 am                                                  | compass points
| WindDir3pm    | Wind direction averaged over 10 minutes prior to 3 pm                                                  | compass points
| WindSpeed9am  | Wind speed averaged over 10 minutes prior to 9 am                                                      | kilometres per hour
| WindSpeed3pm  | Wind speed averaged over 10 minutes prior to 3 pm                                                      | kilometres per hour
| Humidity9am   | Relative humidity at 9 am                                                                              | percent
| Humidity3pm   | Relative humidity at 3 pm                                                                              | percent
| Pressure9am   | Atmospheric pressure reduced to mean sea level at 9 am                                                 | hectopascals
| Pressure3pm   | Atmospheric pressure reduced to mean sea level at 3 pm                                                 | hectopascals
| Cloud9am      | Fraction of sky obscured by cloud at 9 am                                                              | eighths
| Cloud3pm      | Fraction of sky obscured by cloud at 3 pm                                                              | eighths
| Temp9am       | Temperature at 9 am                                                                                    | degrees Celsius
| Temp3pm       | Temperature at 3 pm                                                                                    | degrees Celsius
| RainToday     | The rain for that day was 1mm or more                                                                  | Yes or No
| RainTomorrow  | The rain for that day was 1mm or more. The target variable to predict.                                 | Yes or No

In [ ]:
import pandas as pd
import numpy as np
# magic function - см. Interactive Plotting
%matplotlib widget
from ipywidgets import *
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# путь к папке с данными
data_path = "./data"
# датасет: Rain in Australia: https://www.kaggle.com/datasets/jsphyg/weather-dataset-rattle-package
df = pd.read_csv(Path(data_path, 'weatherAUS.csv'))
df.info()

In [ ]:
df.head()

Отключим ограничение на вывод столбцов датафрейма:

In [ ]:
pd.set_option('display.max_columns', None)
df.describe(include='all')

## 6.4 Библиотека `missingno`: визуальный анализ пропусков в данных

Библиотека [missingno](https://github.com/residentmario/missingno) &ndash; это инструмент Python для визуализации и анализа пропущенных (недостающих) значений в табличных данных `pandas`.

> uv pip install -U missingno

Основные функции:
- Матрица `msno.matrix` &ndash; компактное отображение заполненности данных. Помогает увидеть общую картину пропусков и временные тренды.
- Столбчатая диаграмма `msno.bar` &ndash; простой график, который показывает количество или долю заполненных значений для каждой колонки.
- Тепловая карта `msno.heatmap` &ndash; показывает корреляцию между пропусками в разных столбцах (от $-1$ до $1$).
- Дендрограмма `msno.dendrogram` &ndash; иерархическая кластеризация на базе библиотеки `scipy`, которая объединяет переменные со схожими паттернами отсутствия данных.

Все типы графиков поддерживают следующие параметры:
- `figsize` &ndash; размер фигуры, по умолчанию $(20, 12)$, кроме дендрограмм (высота вычисляется автоматически).
- `fontsize` &ndash; размер шрифта, по умолчанию $16$.
- `labels` &ndash; флаг отображения меток. Для матриц по умолчанию `True` для $\leq 50$ признаков и `False` для $>50$. По умолчанию `True` для дендрограмм и тепловых карт.

In [ ]:
import matplotlib as mpl
import missingno as msno

### `msno.matrix`

In [ ]:
msno.matrix(df, figsize=(12, 5), fontsize=10, label_rotation=45)

Сдвинем ось построения, чтобы метки влезли («in-place»):

In [ ]:
fig = plt.gcf()
fig.subplots_adjust(top=0.80, bottom=0.05)

`plt.gcf()` возвращает объект &ndash; текущую фигуру. `subplots_adjust` меняет `layout` фигуры.  
> Необходимо быть осторожным с применением `plt.gcf()` для библиотек-оберток над `matplotlib`. `missingno` под капотом создает новую фигуру. В Jupyter Notebook при последовательном вызове нескольких графиков `matplotlib` может запутаться в порядке создания объектов. В итоге `plt.gcf()` (*Get Current Figure*) ошибочно цепляет предыдущую (старую) фигуру вместо только что созданной. Поэтому можно явно закрыть текущую фигуру перед построением следующего.

> `msno.matrix` возвращает ось построения, у которой можно получить объект фигуры: `fig = ax.get_figure()`.

Объединим код и зададим цвет заполненных данных.
> `color` принимает $RGB$ триплет в диапазоне $[0.0, 1.0]$, по умолчанию &ndash; $(0.25, 0.25, 0.25)$. Поэтому приведем hex-значение цвета через метод `matplotlib.colors.to_rgb` в $RGB$.

In [ ]:
plt.clf()    # Очищаем текущую активную фигуру
plt.close()  # Закрываем её, чтобы сбросить индекс фигуры в matplotlib
ax = msno.matrix(df, figsize=(12, 5), fontsize=10, label_rotation=45, color=mpl.colors.to_rgb("#004F13"))
fig = ax.get_figure()
fig.subplots_adjust(top=0.80, bottom=0.05)

Справа от матрицы расположен `sparkline`. Он отображает общую динамику заполненности датасета построчно. Когда строка таблицы заполнена полностью, график сдвигается до упора вправо. Если в строке появляются пропуски, линия уходит влево. Слева и справа от этого графика числами подсвечиваются строки с минимальным и максимальным количеством заполненных столбцов в выборке. 
Построение `sparkline` можно отключить: `sparkline=False`.  

> Можно явно передать ось построения в `msno.matrix(ax=my_ax, ...)`, но построение `sparkline` отключится.

> Судя по всему, в `sparkline` индекс строк &ndash; «последний» (*last occurrence*):

In [ ]:
valid_counts = df.notna().sum(axis=1)
v_min, idx_min = valid_counts.min(), valid_counts[::-1].idxmin()
v_max, idx_max = valid_counts.max(), valid_counts[::-1].idxmax()
print(f"min: {v_min} @ row {idx_min}, max: {v_max} @ row {idx_max}")

In [ ]:
df.loc[144260, :].notna()

In [ ]:
df.loc[142302, :].notna()

Вспомогательная функция (вызов перед построением новой фигуры):

In [ ]:
def close_mpl_fig(figure=None, exclude_list: list[plt.figure] = []):
    fig = figure
    if fig is None:
        fig = plt.gcf()
    if fig not in exclude_list:
        plt.clf()    # Очищаем текущую активную фигуру
        plt.close()  # Закрываем её, чтобы сбросить индекс фигуры в matplotlib

### `msno.bar`

Параметры:
- `log` &ndash; включает логарифмический масштаб

In [ ]:
close_mpl_fig()
ax = msno.bar(df, figsize=(12, 5), fontsize=10, label_rotation=45, color=mpl.colors.to_rgb("#004F13"))
fig = ax.get_figure()
fig.subplots_adjust(top=0.85, bottom=0.20)

In [ ]:
close_mpl_fig()
ax = msno.bar(df, figsize=(12, 5), fontsize=10, label_rotation=45, color=mpl.colors.to_rgb("#004F13"), log=True)
fig = ax.get_figure()
fig.subplots_adjust(top=0.85, bottom=0.20)

### `msno.heatmap`

Параметры:
- `cmap` &ndash; цветовая карта

Тепловая карта в `missingno` считает коэффициент корреляции Пирсона для бинарных значений ($0$ &ndash; пропуск, $1$ &ndash; значение заполнено) между всеми парами столбцов:
- $1.0$ &ndash; идеальная прямая зависимость. Если в столбце $А$ есть пропуск, то в столбце $Б$ в этой же строке гарантированно будет пропуск.
- $-1.0$ &ndash; идеальная обратная зависимость. Если в столбце $А$ есть пропуск, то в столбце $Б$ данные обязательно присутствуют.
- $0.0$ &ndash; пропуски в этих переменных никак не связаны друг с другом, они случайны.

In [ ]:
close_mpl_fig()
ax = msno.heatmap(df, figsize=(12, 5), fontsize=10, label_rotation=45)
fig = ax.get_figure()
fig.subplots_adjust(top=0.85, bottom=0.20)

### `msno.dendrogram`

В библиотеке `missingno` функция `msno.dendrogram` строит дерево иерархической кластеризации, которое группирует столбцы по схожести паттернов их заполненности.  
Если тепловая карта `msno.heatmap` показывает связь только между парами колонок, то дендрограмма позволяет увидеть глобальную структуру зависимостей во всем датасете сразу.

График читается снизу вверх (или слева направо, в зависимости от ориентации):
1. Группировка (листья дерева).  
   Колонки, которые соединяются на самых ранних (низких) уровнях, имеют практически идентичное распределение пропусков. Если они соединяются на нулевой отметке, значит, пропуски в них происходят строго в одних и тех же строках.
2. Высота ветвей.  
   Чем выше (дальше) находится узел соединения двух веток, тем меньше связаны пропуски в этих группах столбцов.
3. Полностью заполненные колонки.  
   Столбцы без пропусков (или со схожими паттернами стопроцентной заполненности) моментально объединяются в один большой кластер на самом правом или левом краю графика.

In [ ]:
close_mpl_fig()
ax = msno.dendrogram(df, figsize=(12, 5), fontsize=10, label_rotation=45)
fig = ax.get_figure()
fig.subplots_adjust(top=0.8, bottom=0.0)

## 6.5 Подготовка данных

### Проверка целевого признака `RainTomorrow`

In [ ]:
df['RainTomorrow'].isnull().sum()

Дропнем строки с null.

In [ ]:
df = df.drop(df[df['RainTomorrow'].isna()].index)
df.info()

Проверим баланс классов в целевом признаке:

In [ ]:
df['RainTomorrow'].value_counts()

In [ ]:
df['RainTomorrow'].value_counts() / len(df)

> Что делать с несбалансированными данными: см. [Дисбаланс классов](../../self-study/class_imbalance.md).

### Проверка категориальных признаков
#### Заполним отсутствующие значения

> [pandas.DataFrame.mode](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.mode.html) &ndash; возвращает список наиболее часто встречающихся значений.

Выделим названия категориальных признаков в отдельный массив:

In [ ]:
# cat_cols = [var for var in df.columns if df[var].dtype == 'object']  # pandas < 3.0.0
cat_cols = [var for var in df.columns if df[var].dtype == 'str']  # pandas >= 3.0.0
df[cat_cols].head()

> Про изменение работы со строками в `pandas` 3.0.0 см. [Working with text data @ pandas](https://pandas.pydata.org/docs/user_guide/text.html).

Пропуски в категориальных данных:

In [ ]:
cat_null = df[cat_cols].isnull().sum()
cat_null

Наиболее часто встречающиеся значения в категориальных признаках с наличием пропусков:

In [ ]:
df[cat_null[cat_null > 0].index].mode()

Осуществим замену пропусков:

In [ ]:
for col in cat_cols:
    df.fillna({col: df[col].mode()[0]}, inplace=True)
df[cat_cols].isnull().sum()

#### Мощность признаков (*cardinality*)

Мощность признака &ndash; количество уникальных значений признака.

In [ ]:
len_max = max([len(col) for col in cat_cols])
for col in cat_cols:
    print(f"{col:<{len_max}} labels: {len(df[col].unique())}")

Признак `Date` имеет высокую мощность, что может усложнить задачу классификации. Разобьем дату на составные части: год, месяц, день. Дропнем `Date`.

In [ ]:
df['Date'] = pd.to_datetime(df['Date'])
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df.drop('Date', axis=1, inplace = True)
cat_cols.remove('Date')
df.info()

#### Признак `Location`

In [ ]:
print(f"\"Location\" label count: {len(df.Location.unique())}")
df.Location.unique()

## 6.6 Кодирование признака
> Кодирование категориальных признаков &ndash; преобразование категориальных признаков в численное представление по некоторым правилам.

Гайды:
- [sklearn: Encoding categorical features](https://scikit-learn.org/stable/modules/preprocessing.html#preprocessing-categorical-features)
- [sklearn: LabelEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.LabelEncoder.html#sklearn.preprocessing.LabelEncoder)
- [sklearn: OneHotEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html)
- [pandas: factorize](https://pandas.pydata.org/docs/reference/api/pandas.factorize.html)
- [Хабр: Отличия LabelEncoder и OneHotEncoder в SciKit Learn](https://habr.com/ru/articles/456294/)
- [Хабр: Категориальные признаки](https://habr.com/ru/articles/666234/)

Рассмотрим 2 основных подхода.

### `Label Encoder`  
  
Преобразование представляет собой однозначное соответствие `уникальное значение категориального признака` &harr; `число`, диапазон $[0, N-1]$.
> Главный недостаток `Label Encoder`'a &ndash; создание избыточных зависимостей в данных (*порядок* и *количественное отношение*).

Данный метод должен применяться только для кодирования *целевого категориального признака*, а не входных категориальных признаков (разве что для входных бинарных):  
> Encode target labels with value between 0 and n_classes-1.
> **This transformer should be used to encode *target values*, i.e. y, and not the input X.**

Перекодируем наш признак `Location` (только в качестве примера):
$$
\begin{bmatrix}
    \text{Adelaide} \\[0.3em]
    \text{Albany}   \\[0.3em]
    \text{Albury}   \\[0.3em]
    \cdots           \\[0.3em]
    \text{Woomera}  \\[0.3em]
\end{bmatrix}_{\; 49 \times 1} \quad \rightarrow{} \quad
\begin{bmatrix}
    0 \\[0.3em]
    1 \\[0.3em]
    2 \\[0.3em]
    \vdots \\[0.3em]
    48 \\[0.3em]
\end{bmatrix}_{\; 49 \times 1}
$$
В данном примере `Woomera` в 24 раза "больше" `Albury`.

### `One-Hot Encoder`  

Каждому уникальному значению признака ставится в соответствие бинарный вектор, состоящий из нулей и одной единицы. Каждое значение такого вектора означает принадлежность значения признака одному из уникальных значений (классу или категории):
$$
\begin{matrix}
& \\
\begin{bmatrix} A \\ B \\ C \\ D \end{bmatrix} \quad \rightarrow{} \quad
\left [ \vphantom{ \begin{matrix} 12 \\ 12 \\ 12 \\ 12 \end{matrix} } \right .
\end{matrix}
\hspace{-1.2em}
\begin{matrix}
    A & B & C & D \\
    1 & 0 & 0 & 0 \\
    0 & 1 & 0 & 0 \\
    0 & 0 & 1 & 0 \\
    0 & 0 & 0 & 1
\end{matrix}
\hspace{-0.2em}
\begin{matrix}
& \\
\left . \vphantom{ \begin{matrix} 12 \\ 12 \\ 12 \\ 12 \end{matrix} } \right ]
    \begin{matrix} A \\ B \\ C \\ D \end{matrix}
\end{matrix}
$$
> Главный недостаток `One-Hot Encoder`'a &ndash; генерация избыточного количества данных, вместо одного признака с $N$ уникальными значениями мы получаем $N$ бинарных признаков.

### Кодирование признака `LabelEncoder`'ом с использованием `sklearn`

Пример кодирования LabelEncoder'ом с использованием `sklearn`:

In [ ]:
import sklearn
from sklearn.preprocessing import LabelEncoder
enc = LabelEncoder()
cat_arr = ['Albury', 'BadgerysCreek', 'Cobar', 'CoffsHarbour', 'Moree']
enc.fit(cat_arr)
print(f"Encoded classes: {enc.classes_}")

Преобразование в кодированный вид:

In [ ]:
enc.transform(['BadgerysCreek', 'CoffsHarbour'])

Обратное преобразование из кодированного в исходный вид:

In [ ]:
enc.inverse_transform([0, 2])

### Кодирование признака `LabelEncoder`'ом с использованием `pandas`

Пример кодирования `LabelEncoder`'ом с использованием `pandas`:

In [ ]:
pd.factorize(np.array(cat_arr))

> Явное преобразование в `np.array` необходимо, так как функция принимает объект типов `Series`, `Index`, `ExtensionArray`, `np.ndarray` или `NumpyExtensionArray`.

Или так:

In [ ]:
codes, uniques = pd.factorize(pd.Series(['Albury', 'BadgerysCreek', 'Cobar', 'CoffsHarbour', 'Moree', 'Albury', 'Cobar', 'Cobar']))
print(codes, uniques)

Обратное преобразование:

In [ ]:
restored = uniques.take([0, 2])
restored

### Кодирование признака `OneHotEncoder`'ом с использованием `sklearn`

Пример кодирования `OneHotEncoder`'ом с использованием `sklearn`:

In [ ]:
import sklearn
from sklearn.preprocessing import OneHotEncoder
enc = OneHotEncoder()
enc.fit(np.reshape([cat_arr], (-1, 1)))
print(f"Encoded classes: {enc.categories_}")

In [ ]:
data_new = enc.transform([['BadgerysCreek'], ['CoffsHarbour']])
data_new.toarray()

Обратное преобразование:

In [ ]:
enc.inverse_transform([[0., 0., 1., 0., 0.],
                       [0., 0., 0., 0., 1.]])

### Кодирование признака `OneHotEncoder`'ом с использованием `pandas`

Пример кодирования `OneHotEncoder`'ом с использованием `pandas`:

In [ ]:
cat_dummy = pd.get_dummies(cat_arr)
cat_dummy

Обратное преобразование:

In [ ]:
pd.from_dummies(cat_dummy)

Тестовое кодирование признака `Location`:

In [ ]:
pd.get_dummies(df.Location).head()

## 6.7 Подготовка данных (продолжение)

### Подготовка количественных признаков

Выделим названия количественных признаков в отдельный массив:

In [ ]:
num_cols = [var for var in df.columns if not df[var].dtype == 'str']
df[num_cols].head()

#### Проверка на null

In [ ]:
df[num_cols].isnull().sum()

Посмотрим на диапазон значений (масштаб) количественных признаков.

In [ ]:
print(round(df[num_cols].describe(), 2))

Заполним отсутствующие значения &ndash; заменим медианой.

In [ ]:
for col in num_cols:
    df.fillna({col: df[col].median()}, inplace=True)
df.info()

### Кодирование категориальных признаков

#### Признак `Location`

Перекодируем наш признак с использованием `One-Hot Encoder`.

$$
\begin{bmatrix}
    \text{Adelaide} \\[0.3em]
    \text{Albany}   \\[0.3em]
    \text{Albury}   \\[0.3em]
    \cdots          \\[0.3em]
    \text{Woomera}  \\[0.3em]
\end{bmatrix}_{\; 49 \times 1} \quad \rightarrow{} \quad
\begin{bmatrix}
    1 & 0 & 0 & \cdots & 0 \\[0.3em]
    0 & 1 & 0 & \cdots & 0 \\[0.3em]
    0 & 0 & 1 & \cdots & 0 \\[0.3em]
    \vdots & \vdots & \vdots & \ddots & 0 \\[0.3em]
    0 & 0 & 0 & \cdots & 1 \\[0.3em]
\end{bmatrix}_{\; 49 \times 49}
$$

In [ ]:
df_loc_dummy = pd.get_dummies(df.Location, prefix='Location')
df_loc_dummy

Дропаем `Location` и добавляем его закодированный вариант:

In [ ]:
df = df.drop('Location', axis = 1)
df = df.join(df_loc_dummy)
df.info(max_cols=20)

Оставшиеся категориальные признаки:

In [ ]:
cat_left = [var for var in df.columns if df[var].dtype == 'str']
cat_left

#### Кодирование категориальных признаков направления ветра `OneHotEncoder`'ом

In [ ]:
cat_left = ['WindGustDir', 'WindDir9am', 'WindDir3pm']
df = pd.get_dummies(data=df, columns=cat_left)
df.info(max_cols=20)

Список признаков:

In [ ]:
list(df.columns)

#### Преобразование бинарных признаков `RainToday` и `RainTomorrow`

Вручную заменим классы бинарных признаков на численное представление и приведем тип данных к `int`:

In [ ]:
df.replace({'RainToday': {'No': '0', 'Yes': '1'}}, inplace = True)
df.replace({'RainTomorrow': {'No': '0', 'Yes': '1'}}, inplace = True)
df['RainToday'] = df['RainToday'].astype(int)
df['RainTomorrow'] = df['RainTomorrow'].astype(int)

In [ ]:
df.corr()

### Выбросы (outliers)

Расширим список численных признаков &ndash; добавим `RainToday`:

In [ ]:
import copy
num_cols_ext = copy.deepcopy(num_cols)
num_cols_ext.append('RainToday')

Построим `boxplot`'ы:

In [ ]:
close_mpl_fig()
fig, ax = plt.subplots(figsize=(11, 5))
sns.boxplot(data = df[num_cols_ext], ax = ax)
plt.subplots_adjust(bottom=0.2)
plt.xticks(rotation=90)
plt.show()

Из графика видно, что данные в наборе у нас разных масштабов. Необходимо нормализовать данные.

### Нормализация признаков

Базовые методы нормализации:  
- **MinMax** нормализация &ndash; приведение данных к масштабу $[0, 1]$ &ndash; [sklearn.preprocessing.MinMaxScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html) в `sklearn`.
- **Стандартная** нормализация &ndash; данные имеют среднее $0$ и стандартное отклонение $1$ &ndash; [sklearn.preprocessing.StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) в `sklearn`.

Посмотрим еще раз на численные данные:

In [ ]:
df[num_cols_ext].describe().T

Нормализуем данные с использованием `MinMaxScaler`:

In [ ]:
all_cols = list(df.columns)
mm_scaler = sklearn.preprocessing.MinMaxScaler()
features_scaled = mm_scaler.fit_transform(df[all_cols])
df_scaled = pd.DataFrame(features_scaled, columns=all_cols)
df_scaled

In [ ]:
df_scaled.describe()

### Выбросы (outliers) после нормализации

In [ ]:
close_mpl_fig()
fig, ax = plt.subplots(figsize=(11, 5))
sns.boxenplot(data = df_scaled[num_cols_ext], ax = ax)
plt.subplots_adjust(bottom=0.2)
plt.xticks(rotation=90)
plt.show()

### Подправим outliers

Определим границы ящика с усами.

In [ ]:
def get_bounds(dataframe, col):
    iqr = dataframe[col].quantile(0.75) - dataframe[col].quantile(0.25)
    lower_bound = dataframe[col].quantile(0.25) - 1.5 * iqr
    upper_bound = dataframe[col].quantile(0.75) + 1.5 * iqr
    return lower_bound, upper_bound

In [ ]:
num_cols_clean = ['MinTemp', 'MaxTemp', 'Rainfall', 'Evaporation', 'Sunshine', 'WindGustSpeed', 'WindSpeed9am', 'WindSpeed3pm', 'Humidity9am', 'Humidity3pm', 'Pressure9am', 'Pressure3pm', 'Cloud9am', 'Cloud3pm', 'Temp9am', 'Temp3pm']

bounds_dict = dict()

for col in num_cols_clean:
    lb, ub = get_bounds(df_scaled, col)
    bounds_dict[col] = [lb, ub]
    print(f"{col:<13} outliers are values < {lb:.2f} or > {ub:.2f}")

In [ ]:
df[num_cols_clean].count()

Удалим выбросы, лежащие за пределами интерквантильного расстояния:

```python
Q1 = df['y'].quantile(0.25)
Q3 = df['y'].quantile(0.75)
IQR = Q3 - Q1

# 2. Define the acceptable boundaries
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# 3. Keep only the data within the boundaries
df_cleaned = df[(df['y'] >= lower_bound) & (df['y'] <= upper_bound)]
```

In [ ]:
from copy import deepcopy

def clean_data(df, bounds_dict: dict):
    df_clean = deepcopy(df)
    print(df_clean.shape)

    for k, v in bounds_dict.items():
        arr = np.array((df_clean[k] > v[0]) & (df_clean[k] < v[1])).reshape((-1, 1))
        print(f"{k}: bounds: {v}")
        print(f"  old: {df_clean[k].shape[0]}, new: {np.count_nonzero(arr)}, diff: {np.count_nonzero(arr) - df_clean[k].shape[0]}")
        df_clean = df_clean[(df_clean[k] > v[0]) & (df_clean[k] < v[1])]
    return df_clean

df_clean = clean_data(df_scaled, bounds_dict)

In [ ]:
df_clean.info()

Сравним до и после:

In [ ]:
close_mpl_fig()
fig, ax = plt.subplots(2, 1, figsize=(11, 5))
sns.boxenplot(data = df_scaled[num_cols_ext], ax = ax[0])
ax[0].set_xticks([])
ax[0].title.set_text('Before')
sns.boxenplot(data = df_clean[num_cols_ext], ax = ax[1])
ax[1].title.set_text('After')
plt.subplots_adjust(bottom=0.2)
plt.xticks(rotation=90)
fig.tight_layout()
plt.show()

Выделим $X$ и $y$ из набора данных:

In [ ]:
X = df_clean.drop(['RainTomorrow'], axis=1)
y = df_clean['RainTomorrow']
X.shape, y.shape

### 6.8 Обучение и предсказание

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
model_knn = KNeighborsClassifier(n_neighbors=3)

### Разделение выборки на обучающую и тестовую

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 42)
print(y_train.shape, y_test.shape)

### Обучение модели

In [ ]:
model_knn.fit(X_train, y_train)

## 6.9 Оценка качества модели

Получим предсказание модели на *тестовой выборке*:

In [ ]:
y_pred = model_knn.predict(X_test)

Вычислим ручками:

In [ ]:
print(f"Test accuracy: {np.mean( y_pred == y_test ):.2f}")

Методом `score`:

In [ ]:
model_knn.score(X_test, y_test)

`accuracy_score` в `sklearn`:

In [ ]:
sklearn.metrics.accuracy_score(y_pred, y_test)

### Точность модели в зависимости от значения гиперпараметра

In [ ]:
training_accuracy = []
test_accuracy = []
neighbors_settings = range(1, 16)
for n_neighbors in neighbors_settings:
  clf = KNeighborsClassifier(n_neighbors = n_neighbors)
  clf.fit(X_train, y_train)
  training_accuracy.append(clf.score(X_train, y_train))
  test_accuracy.append(clf.score(X_test, y_test))

Построим график:

In [ ]:
close_mpl_fig()
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(neighbors_settings, training_accuracy, label="train accuracy")
ax.plot(neighbors_settings, test_accuracy, label="test accuracy")
ax.set_ylabel("Accuracy")
ax.set_xlabel("Neighbors count")
ax.set_xticks(range(1, 16))
ax.legend()

### Кросс-валидация (перекрёстная проверка)

Перекрестная проверка (кросс-валидация) представляет собой статистический метод оценки обобщающей способности, который является более устойчивым и точным, чем разбиение данных на обучающий и тестовый наборы. В перекрестной проверке данные разбиваются несколько раз и строится несколько моделей. Наиболее часто используемый вариант перекрестной проверки &ndash; $k$-блочная кросс-валидация ($k$-fold cross-validation), в которой $k$ &ndash; это задаваемое пользователем число, как правило, $5$ или $10$.

<div align="center">
  <img src="images/k_fold.svg" width="1000" title="Кросс-валидация"/>
  <p style="text-align: center">
    Рисунок &ndash; Кросс-валидация
  </p>
</div>

> Пример умозрителен: выборка отсортирована по меткам класса, разбиение (тестовая выборка) последовательное.

В `scikit-learn` перекрестная проверка реализована с помощью функции [sklern.model_selection.cross_val_score](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html).

In [ ]:
clf = sklearn.neighbors.KNeighborsClassifier(n_neighbors = 3)
scores = sklearn.model_selection.cross_val_score(clf, X, y, cv = 3)
print(f"Scores: {scores}\nAvg score: {scores.mean()}")

In [ ]:
clf = sklearn.neighbors.KNeighborsClassifier(n_neighbors = 3)
scores = sklearn.model_selection.cross_val_score(clf, X, y, cv = 5)
print(f"Scores: {scores}\nAvg score: {scores.mean()}")

In [ ]:
knn_k_list = range(1, 16)
cv_k = 10
for knn_k in knn_k_list:
    clf = sklearn.neighbors.KNeighborsClassifier(n_neighbors = knn_k)
    scores = sklearn.model_selection.cross_val_score(clf, X, y, cv = cv_k)
    print(f"Neighbors: {knn_k}, folds count: {cv_k} avg score: {scores.mean()}")

### Стратифицированная $k$-блочная кросс-валидация

<div align="center">
  <img src="images/stratified_k_fold.svg" width="1000" title="Стратифицированная k-блочная кросс-валидация"/>
  <p style="text-align: center">
    Рисунок &ndash; Стратифицированная k-блочная кросс-валидация
  </p>
</div>

В стратифицированной перекрестной проверке мы разбиваем данные таким образом, чтобы пропорции классов в каждом блоке в точности соответствовали пропорциям классов в наборе данных.  
[Stratified K-fold @ sklearn](https://scikit-learn.org/stable/modules/cross_validation.html#stratified-k-fold).
> Пример умозрителен: выборка отсортирована по меткам класса, разбиение (тестовая выборка) последовательное.

In [ ]:
knn_k_list = range(1, 16)
cv_k = 10
cvgen = sklearn.model_selection.StratifiedKFold(cv_k)
for knn_k in knn_k_list:
    clf = sklearn.neighbors.KNeighborsClassifier(n_neighbors = knn_k)
    scores = sklearn.model_selection.cross_val_score(clf, X, y, cv = cvgen)
    print(f"Neighbors: {knn_k}, stratified folds count: {cv_k} avg score: {scores.mean()}")

## 6.10 Метрики оценки точности бинарной классификации

В случае бинарной классификации мы говорим о *положительном* (*positive*) классе и *отрицательном* (*negative*) классе, подразумевая под положительным классом интересующий нас класс.

Ошибка первого рода ($\alpha$-ошибка, *ложноположительное заключение*) &ndash; ситуация, когда отвергнута верная нулевая гипотеза (об отсутствии связи между явлениями или искомого эффекта).  
Ошибка второго рода ($\beta$-ошибка, *ложноотрицательное заключение*) &ndash; ситуация, когда принята неверная нулевая гипотеза.  

Матрица ошибок (*confusion matrix*) показывает количество верных и неверных предсказаний модели по сравнению с реальными данными.

<div align="center">
<table style = "border: 1px solid black;">
    <caption style="padding-bottom: 10px;">Confusion matrix</caption>
    <tr style = "border: 1px solid black;">
        <th rowspan=2 colspan=2 style = "border: 1px solid black;"></th>
        <th colspan=2 style = "border: 1px solid black;">Верная гипотеза</th>
    </tr>
    <tr style = "border: 1px solid black; background-color: rgba(255, 255, 255, 0.5);">
        <th style = "border: 1px solid black;">H<sub>0</sub></th>
        <th style = "border: 1px solid black;">H<sub>1</sub></th>
    </tr>
    <tr style = "border: 1px solid black;">
        <th rowspan=2 style = "border: 1px solid black;  background-color: rgba(255, 255, 255, 0.25);">Результат применения критерия</th>
        <th style = "border: 1px solid black; background-color: rgb(255, 255, 255, 0.25);">H<sub>0</sub></th>
        <td style = "border: 1px solid black; background-color: rgba(0, 255, 0, 0.25); text-align: center;"><b>H<sub>0</sub></b> верно принята</td>
        <td style = "border: 1px solid black; background-color: rgba(255, 0, 0, 0.25); text-align: center;"><b>H<sub>0</sub></b> неверно принята<br>(ошибка второго рода)</td>
    </tr>
    <tr style = "border: 1px solid black;">
        <th style = "border: 1px solid black; background-color: rgb(255, 255, 255, 0.25);">H<sub>1</sub></th>
        <td style = "border: 1px solid black; background-color: rgba(255, 0, 0, 0.25); text-align: center;"><b>H<sub>0</sub></b> неверно отвергнута<br>(ошибка первого рода)</td>
        <td style = "border: 1px solid black; background-color: rgba(0, 255, 0, 0.25); text-align: center;"><b>H<sub>0</sub></b> верно отвергнута</td>
    </tr>
</table>
<br>
<table style = "border: 1px solid black; font-size: 42px;">
    <tr>
        <td style = "background-color: rgba(0, 255, 0, 0.25);">
            TP
        </td>
        <td style = "background-color: rgba(255, 0, 0, 0.25);">
            FN
        </td>
    </tr>
    <tr>
        <td style = "background-color: rgba(255, 0, 0, 0.25);">
            FP
        </td>
        <td style = "background-color: rgba(0, 255, 0, 0.25);">
            TN
        </td>
    </tr>
</table>
</div>

Можем посчитать вручную:

In [ ]:
def get_confusions(y_gt, y_pred):
    cm = sklearn.metrics.confusion_matrix(y_gt, y_pred)
    # tp fn fp tn
    return cm[1, 1], cm[1, 0], cm[0, 1], cm[0, 0]

In [ ]:
clf = sklearn.neighbors.KNeighborsClassifier(n_neighbors = 15)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
tp, fn, fp, tn = get_confusions(y_test, y_pred)
cm = [[tp, fn], [fp, tn]]

In [ ]:
print(f"{cm[0]}\n{cm[1]}")

Графически:

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

close_mpl_fig()
display = ConfusionMatrixDisplay.from_estimator(
    clf, X_test, y_test, display_labels=clf.classes_, values_format='d'
)
_ = display.ax_.set_title("Confusion matrix")
display.ax_.set_aspect('equal', adjustable='box')

### Accuracy (правильность)

$$
Accuracy = \frac{ TP + TN  }{ TP + TN + FP + FN}
$$

Другими словами, правильность &ndash; это количество верно классифицированных примеров $TP$ и $TN$, поделенное на общее количество примеров.

Ручками:

In [ ]:
def get_accuracy(tp, fn, fp, tn):
    return (tp + tn) / (tp + tn + fp + fn)

In [ ]:
print(f"Accuracy: {get_accuracy(tp, fn, fp, tn)}")

`accuracy_score` в `sklearn`:

In [ ]:
sklearn.metrics.accuracy_score(y_test, y_pred)

### Precision (точность)

$$
Precision = \frac{ TP }{ TP + FP }
$$

Точность показывает, насколько можно доверять классификатору в случае срабатывания: сколько из предсказанных положительных примеров оказались действительно положительными.

Таким образом, точность &ndash; это доля истинно положительных примеров от общего количества предсказанных положительных примеров.

Точность используется в качестве показателя качества модели, когда цель состоит в том, чтобы снизить количество ложно положительных примеров.

Точность также известна как *прогностическая ценность положительного результата* (*positive predictive value*, *PPV*).

Ручками:

In [ ]:
def get_precision(tp, fp):
    return tp / (tp + fp)

In [ ]:
print(f"Precision: {get_precision(tp, fp)}")

`precision_score` в `sklearn`:

In [ ]:
sklearn.metrics.precision_score(y_test, y_pred)

### Recall (полнота)

$$
Recall = \frac{ TP }{ TP + FN }
$$

*Полнота* (*recall*) показывает, на какой доле истинных объектов алгоритм срабатывает.
Другие названия полноты: *чувствительность* (*sensitivity*), *процент результативных ответов* или *хит-рейт* (*hit rate*) или *доля истинно положительных ответов* (*true positive rate*, *TPR*).

Полнота используется в качестве показателя качества модели, когда нам необходимо определить все положительные примеры, то есть, когда важно снизить количество ложно отрицательных примеров.

Нужно искать компромисс между оптимизацией полноты и оптимизацией точности. Можно получить идеальную полноту, спрогнозировав все примеры как положительные &ndash; не будет никаких ложно отрицательных и истинно отрицательных примеров. Однако прогнозирование всех примеров как положительных приведет к большому количеству ложно положительных примеров, и, следовательно, точность будет очень низкой.  

Ручками:

In [ ]:
def get_recall(tp, fn):
    return tp / (tp + fn)

In [ ]:
print(f"Recall: {get_recall(tp, fn)}")

### Интегральные метрики на основе *точности* и *полноты*
#### Арифметическое среднее
Единая метрика может быть получена как арифметическое среднее точности и полноты:
$$
A = \frac{1}{2} \cdot (precision + recall)
$$

Кривая точности-полноты (*Precision-Recall Curve*):

In [ ]:
from sklearn.metrics import PrecisionRecallDisplay

close_mpl_fig()
display = PrecisionRecallDisplay.from_estimator(
    clf, X_test, y_test, name="kNN"
)
_ = display.ax_.set_title("2-class Precision-Recall curve")
display.ax_.set_aspect('equal', adjustable='box')

Эта кривая особенно полезна для оценки моделей классификации в условиях сильного дисбаланса классов (когда одного класса гораздо больше, чем другого).

Главное преимущество &ndash; автоматический расчет $AP$ (*average precision*). На графике сразу отображается метрика $AP$, которая показывает площадь под кривой (чем ближе к $1.0$, тем лучше модель).

Вариант с использованием предсказанных вероятностей классов:

In [ ]:
from sklearn.metrics import PrecisionRecallDisplay

probabilities = clf.predict_proba(X_test)
positive_class_probs = probabilities[:, 1]

close_mpl_fig()
display = PrecisionRecallDisplay.from_predictions(
    y_test, positive_class_probs, name="kNN"
)
_ = display.ax_.set_title("2-class Precision-Recall curve")
display.ax_.set_aspect('equal', adjustable='box')

> `PrecisionRecallDisplay` на вход ожидает вероятность только положительного класса (класса $1$). Метод `predict_proba` возвращает вероятности для всех классов, поэтому из него нужно явно взять вторую колонку [:, 1].

### F-мера (f-measure)

Одним из способов подытожить их является *F-мера* (*F-measure*), которая представляет собой гармоническое среднее точности и полноты:

$$
F = 2 \cdot \frac{precision \cdot recall}{precision + recall}
$$

 Она стремится к нулю, если точность или полнота стремится к нулю.

Если необходимо отдать предпочтение точности или полноте, следует использовать расширенную F-меру, в которой есть параметр $\beta$:

$$
F = (1 + \beta^2) \cdot \frac{ precision \cdot recall }{ \beta^2 \cdot precision + recall }
$$

где $\beta$ принимает значения в диапазоне $0 < \beta < 1$, если вы хотите отдать приоритет точности, а при $\beta > 1$ приоритет отдается полноте. При $\beta = 1$ формула сводится к предыдущей и вы получаете сбалансированную F-меру (также ее называют F1).

*f-мера* действительно дает более лучшее представление о качестве модели, чем правильность. Однако, в отличие от *правильности*, ее труднее интерпретировать и объяснить.

### `classification_report`

In [ ]:
print(sklearn.metrics.classification_report(y_test, y_pred, target_names=["No rain tomorrow", "Rain tomorrow"]))

Точность вывода можно варьировать:

In [ ]:
print(sklearn.metrics.classification_report(y_test, y_pred, target_names=["No rain tomorrow", "Rain tomorrow"], digits=7))

Обертка в `DataFrame` со стилизацией:

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

def classification_report_prettified(y_true, y_pred, *, labels=None, target_names=None, sample_weight=None, digits=2, output_dict=True, zero_division='warn', title=None):
    crp = pd.DataFrame(classification_report(y_true, y_pred, labels=labels, target_names=target_names, sample_weight=sample_weight, digits=digits, output_dict=output_dict, zero_division=zero_division)).transpose()
    crp['support'] = crp['support'].apply(int)
    crp_styled = crp.style
    if title is not None:
        styles = [
            dict(selector="caption",
                 props=[("font-weight", "bold"), 
                        ("text-align", "center"),
                        ("margin-bottom", "10px")])
        ]
        crp_styled = crp_styled.set_caption(title).set_table_styles(styles)
    crp_styled = crp_styled.apply_index(lambda v: ['font-weight: bold' for v_ in v], axis="index")
    return crp_styled

In [ ]:
classification_report_prettified(y_test, y_pred, target_names=["No rain tomorrow", "Rain tomorrow"], digits=7)

### Оценка принадлежности к классу, значения порога

Многие алгоритмы бинарной классификации устроены следующим образом: сначала вычисляется некоторое вещественное число $ b(x) $, которое сравнивается с порогом $ t $.

$$
a(x) = [b(x) > t],
$$

где $ b(x) $ — оценка принадлежности "положительному" классу.

Часто бывает необходимо оценить качество именно оценки принадлежности, а порог выбирается позже из соображений на точность или полноту.

<div align="center">
  <img src="images/CreditCardFraudDetectionClassifier_1.png" width="674" title="CreditCardFraudDetectionClassifier"/>
</div>

Пусть рассматривается задачного кредитного скоринга и была построена некоторая функция $b(x)$, которая оценивает вероятность возврата кредита клиентом $x$. Далее классификатор строится следующим образом:

$$
a(x) = [b(x) > 0.5]
$$

При этом получилось, что точность ($precision$) равна 10%, а полнота ($recall$) &ndash; 70%. Это очень плохой алгоритм, так как 90% клиентов, которым будет выдан кредит, не вернут его.

$$
\begin{matrix}
    Recall = \frac{ TP }{ TP + FN } \\\\
    Precision = \frac{ TP }{ TP + FP }
\end{matrix}
$$

При этом непонятно, в чем дело: был плохо выбран порог или алгоритм не подходит для решения данной задачи. Именно для этого необходимо измерять качество самих оценок $b(x)$.

<div align="center">
  <img src="images/mlconcepts_image2.png" width="607" title="CreditCardFraudDetectionClassifier"/>
</div>

### ROC-кривая

Еще один инструмент, который обычно используется для анализа поведения классификаторов при различных пороговых значениях &ndash; это кривая *рабочей характеристики приемника* (*receiver operating characteristics curve*) или кратко *ROC-кривая* (*ROC curve*). Как и *кривая точности-полноты*, *ROC-кривая* позволяет рассмотреть все пороговые значения для данного классификатора, но вместо точности и полноты она показывает *долю ложно положительных примеров* (*false positive rate*, $FPR$) в сравнении с *долей истинно положительных примеров* (*true positive rate*, $TPR$).

$$
\begin{matrix}
    TPR = \frac{ TP }{ TP + FN } \\\\
    FPR = \frac{ FP }{ FP + TN }
\end{matrix}
$$

*Доля истинно положительных примеров* &ndash; это просто еще одно название *полноты*, тогда как *доля ложно положительных примеров* &ndash; это доля ложно положительных примеров от общего количества отрицательных примеров.

ROC-кривая строится строится в осях *False Positive Rate* (ось $X$) и *True Positive Rate* (ось $Y$), аналогично $PR$-кривой: постепенно рассматриваются случаи различных значений порогов и отмечаются точки на графике.

<div align="center">
  <img src="https://bitbucket.org/despairr/ds-course-2018/raw/c3f5df9d66d44fe424b4a4f0c0a0195f0ff9b0f7/intro-to-ml-images/1112.png" width="540" title="ROC"/>
  <br>
  <img src="https://bitbucket.org/despairr/ds-course-2018/raw/c3f5df9d66d44fe424b4a4f0c0a0195f0ff9b0f7/intro-to-ml-images/1113.png" width="540" title="ROC"/>
</div>

Кривая стартует с точки $(0, 0)$ и приходит в точку $(1, 1)$. При этом, если существует идеальный классификатор, кривая должна пройти через точку $(0, 1)$. Чем ближе кривая к этой точке, тем лучше будут оценки, а площадь под кривой будет характеризовать качество оценок принадлежности к первому классу. Такая метрика называется $\textit{AUC-ROC}$, или площадь под $ROC$-кривой.

В случае идеального алгоритма $\textit{AUC-ROC} = 1$, а в случае худшего $\textit{AUC-ROC} = \frac{1}{2}$. Значение $\textit{AUC-ROC}$ имеет смысл вероятности того, что если были выбраны случайный положительный и случайный отрицаельный объекты выборки, положительный объект получит оценку принадлежности выше, чем отрицательный объект.

`roc_auc_score` в `sklearn`:

In [ ]:
from sklearn.metrics import roc_auc_score
roc_auc_score(y_test, clf.predict_proba(X_test)[:, 1])

`RocCurveDisplay.from_estimator()`: 

In [ ]:
from sklearn.metrics import RocCurveDisplay

close_mpl_fig()
display = RocCurveDisplay.from_estimator(
    clf, X_test, y_test, name="kNN"
)
_ = display.ax_.set_title("ROC curve")
display.ax_.set_aspect('equal', adjustable='box')

`RocCurveDisplay.from_predictions`:

In [ ]:
from sklearn.metrics import RocCurveDisplay

probabilities = clf.predict_proba(X_test)
positive_class_probs = probabilities[:, 1]

close_mpl_fig()
display = RocCurveDisplay.from_predictions(
    y_test, positive_class_probs, name="kNN"
)
_ = display.ax_.set_title("ROC curve")
display.ax_.set_aspect('equal', adjustable='box')